# Wyckoff ChatBot - LLaMA Fine-Tuning (Colab Pro)

## GPU Selection Guide:
1. Go to **Runtime → Change runtime type**
2. Select **GPU** as Hardware accelerator
3. Choose GPU type:
   - **A100** (Best - 40GB) ← Use if available
   - **V100** (Good - 16GB)
   - **T4** (Works - 15GB)

## Run Order:
1. Run all setup cells first
2. Upload your CSV data
3. Run training
4. Test the model
5. Save to Google Drive

---
## STEP 1: Check GPU & Install Dependencies

In [1]:
# Check GPU
!nvidia-smi

import torch
print(f"\n PyTorch version: {torch.__version__}")
print(f" CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f" GPU: {torch.cuda.get_device_name(0)}")
    print(f" GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Sun Nov 30 23:03:07 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import torch


!pip install -q --upgrade bitsandbytes

!pip install -q transformers==4.36.0
!pip install -q peft==0.7.0
!pip install -q accelerate==0.25.0
!pip install -q datasets==2.15.0
!pip install -q trl==0.7.4
!pip install -q sentence-transformers==2.2.2
!pip install -q chromadb==0.4.18

print("\n All packages installed!")


 All packages installed!


In [15]:
!pip install tokenizers==0.15.0 transformers==4.36.0 --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 12.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Create project folder
!mkdir -p /content/drive/MyDrive/wyckoff_chatbot/models
!mkdir -p /content/drive/MyDrive/wyckoff_chatbot/data

print(" Google Drive mounted!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Google Drive mounted!


In [10]:
# HuggingFace Login (required for LLaMA)
# Get token from: https://huggingface.co/settings/tokens

from huggingface_hub import login

# Enter your HuggingFace token here
HF_TOKEN = "your token here"

login(token=HF_TOKEN)
print(" Logged in to HuggingFace!")

 Logged in to HuggingFace!


---
## STEP 2: Upload & Prepare Data

In [5]:
from google.colab import files

print(" Upload your wyckoff_qa_all.csv file:")
uploaded = files.upload()

# Get filename
csv_filename = list(uploaded.keys())[0]
print(f"\n Uploaded: {csv_filename}")

 Upload your wyckoff_qa_all.csv file:


Saving wyckoff_all_labels_combined.csv to wyckoff_all_labels_combined (2).csv

 Uploaded: wyckoff_all_labels_combined (2).csv


In [ ]:
# Option 2: Use file from Google Drive (if already uploaded)
# csv_filename = "/content/drive/MyDrive/wyckoff_chatbot/data/wyckoff_qa_all.csv"

In [6]:
# Load and inspect data
import pandas as pd

df = pd.read_csv(csv_filename)
print(f" Loaded {len(df)} Q&A pairs")
print(f"\nColumns: {list(df.columns)}")
print(f"\nLabel distribution:")
if 'Label' in df.columns:
    print(df['Label'].value_counts())

print(f"\n Sample Q&A:")
print(f"Q: {df.iloc[0]['Questions'][:100]}...")
print(f"A: {df.iloc[0]['Answers'][:100]}...")

 Loaded 1230 Q&A pairs

Columns: ['Questions', 'Answers', 'Label']

Label distribution:
Label
Strategy Development    216
Psychology              210
Timing                  202
Personal Life           201
Risk Management         201
Adaptability            200
Name: count, dtype: int64

 Sample Q&A:
Q: When was Richard Wyckoff born?...
A: Richard Demille Wyckoff was born in 1873....


In [7]:
# Format data for training
def format_prompt(question, answer):
    """Format Q&A into instruction format for LLaMA."""
    return f"""### Instruction:
You are an expert on Richard Wyckoff's trading methodology. Answer the following question accurately and thoroughly.

### Question:
{question}

### Answer:
{answer}"""

# Create formatted dataset
df['text'] = df.apply(lambda row: format_prompt(row['Questions'], row['Answers']), axis=1)

# Shuffle and split
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
split_idx = int(len(df) * 0.9)  # 90% train, 10% val

train_df = df[:split_idx]
val_df = df[split_idx:]

print(f"✅ Train: {len(train_df)}, Validation: {len(val_df)}")
print(f"\n📋 Sample formatted prompt:")
print(train_df.iloc[0]['text'][:500])

✅ Train: 1107, Validation: 123

📋 Sample formatted prompt:
### Instruction:
You are an expert on Richard Wyckoff's trading methodology. Answer the following question accurately and thoroughly.

### Question:
How does the Point and Figure count influence timing of profit targets during markup?

### Answer:
P&F counts from accumulation project price objectives that help time partial exits. As price approaches the first objective, timing shifts toward protecting profits. Stepping-stone confirmations during markup validate original timing and suggest contin


---
## STEP 3: Load Model with Optimal GPU Settings

In [8]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
import gc

# Clear GPU memory
gc.collect()
torch.cuda.empty_cache()


# Detect GPU and set optimal config
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0

print(f" GPU: {gpu_name}")
print(f" Memory: {gpu_memory:.1f} GB")

# Auto-configure based on GPU
if "A100" in gpu_name:
    CONFIG = {
        "model_name": "meta-llama/Llama-2-7b-hf",
        "batch_size": 4,
        "gradient_accumulation": 4,
        "max_length": 512,
        "lora_r": 32,
        "lora_alpha": 64,
        "use_4bit": True,
        "num_epochs": 3,
    }
    print("⚡ A100 detected - Using HIGH performance settings")

elif "V100" in gpu_name or gpu_memory >= 15:
    CONFIG = {
        "model_name": "meta-llama/Llama-2-7b-hf",
        "batch_size": 2,
        "gradient_accumulation": 8,
        "max_length": 512,
        "lora_r": 16,
        "lora_alpha": 32,
        "use_4bit": True,
        "num_epochs": 3,
    }
    print("⚡ V100/16GB detected - Using MEDIUM performance settings")

else:  # T4 or smaller
    CONFIG = {
        "model_name": "meta-llama/Llama-2-7b-hf",
        "batch_size": 1,
        "gradient_accumulation": 16,
        "max_length": 384,
        "lora_r": 8,
        "lora_alpha": 16,
        "use_4bit": True,
        "num_epochs": 3,
    }
    print("⚡ T4/smaller detected - Using OPTIMIZED settings")

# Common settings
CONFIG["learning_rate"] = 2e-4
CONFIG["output_dir"] = "/content/drive/MyDrive/wyckoff_chatbot/models/llama_wyckoff_lora"
CONFIG["lora_dropout"] = 0.05

print(f"\n Config: {CONFIG}")

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


 GPU: NVIDIA A100-SXM4-40GB
 Memory: 42.5 GB
⚡ A100 detected - Using HIGH performance settings

 Config: {'model_name': 'meta-llama/Llama-2-7b-hf', 'batch_size': 4, 'gradient_accumulation': 4, 'max_length': 512, 'lora_r': 32, 'lora_alpha': 64, 'use_4bit': True, 'num_epochs': 3, 'learning_rate': 0.0002, 'output_dir': '/content/drive/MyDrive/wyckoff_chatbot/models/llama_wyckoff_lora', 'lora_dropout': 0.05}


In [11]:
# Load tokenizer
print(f" Loading tokenizer: {CONFIG['model_name']}")

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG['model_name'],
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(" Tokenizer loaded!")

 Loading tokenizer: meta-llama/Llama-2-7b-hf


tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

✅ Tokenizer loaded!


In [17]:

gc.collect()
torch.cuda.empty_cache()

print(f" Loading model: {CONFIG['model_name']}")

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load model WITH FIXES
model = AutoModelForCausalLM.from_pretrained(
    CONFIG['model_name'],
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
    attn_implementation="eager",
)

# Disable cache (causes issues with gradient checkpointing)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Prepare for training
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

print(" Model loaded!")
print(f" GPU Memory Used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

 Loading model: meta-llama/Llama-2-7b-hf


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

 Model loaded!
 GPU Memory Used: 12.78 GB


In [18]:
# Apply LoRA
print(" Applying LoRA adapters...")

lora_config = LoraConfig(
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("\n LoRA applied!")

 Applying LoRA adapters...
trainable params: 79,953,920 || all params: 6,818,369,536 || trainable%: 1.172625208678628

 LoRA applied!


---
## STEP 4: Prepare Dataset

In [19]:
from tqdm import tqdm

def tokenize_data(texts, tokenizer, max_length):
    """Tokenize list of texts."""
    tokenized = []

    for text in tqdm(texts, desc="Tokenizing"):
        tokens = tokenizer(
            text,
            truncation=True,
            max_length=max_length,
            padding="max_length",
            return_tensors=None
        )
        tokens["labels"] = tokens["input_ids"].copy()
        tokenized.append(tokens)

    return Dataset.from_dict({
        "input_ids": [t["input_ids"] for t in tokenized],
        "attention_mask": [t["attention_mask"] for t in tokenized],
        "labels": [t["labels"] for t in tokenized]
    })

# Tokenize
print(" Tokenizing training data...")
train_dataset = tokenize_data(train_df['text'].tolist(), tokenizer, CONFIG['max_length'])

print(" Tokenizing validation data...")
val_dataset = tokenize_data(val_df['text'].tolist(), tokenizer, CONFIG['max_length'])

print(f"\n Train dataset: {len(train_dataset)} samples")
print(f" Val dataset: {len(val_dataset)} samples")

 Tokenizing training data...


Tokenizing: 100%|██████████| 1107/1107 [00:00<00:00, 5102.21it/s]


 Tokenizing validation data...


Tokenizing: 100%|██████████| 123/123 [00:00<00:00, 5568.38it/s]


 Train dataset: 1107 samples
 Val dataset: 123 samples


---
## STEP 5: Train the Model

In [20]:
import os
os.makedirs(CONFIG['output_dir'], exist_ok=True)

# Training arguments optimized for Colab
training_args = TrainingArguments(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_epochs'],
    per_device_train_batch_size=CONFIG['batch_size'],
    per_device_eval_batch_size=CONFIG['batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation'],
    learning_rate=CONFIG['learning_rate'],
    warmup_ratio=0.1,
    logging_steps=10,
    save_steps=50,
    eval_steps=50,
    evaluation_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    save_total_limit=3,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
    dataloader_pin_memory=False,
    remove_unused_columns=False,
)

# Data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print(" Trainer configured!")
print(f"\n Training config:")
print(f"   Epochs: {CONFIG['num_epochs']}")
print(f"   Batch size: {CONFIG['batch_size']}")
print(f"   Gradient accumulation: {CONFIG['gradient_accumulation']}")
print(f"   Effective batch size: {CONFIG['batch_size'] * CONFIG['gradient_accumulation']}")

 Trainer configured!

 Training config:
   Epochs: 3
   Batch size: 4
   Gradient accumulation: 4
   Effective batch size: 16


In [21]:
print(f"\n Estimated time: {len(train_dataset) * CONFIG['num_epochs'] / (CONFIG['batch_size'] * CONFIG['gradient_accumulation']) * 2 / 60:.0f} minutes")


trainer.train()


print(" TRAINING COMPLETE!")



 Estimated time: 7 minutes


Step,Training Loss,Validation Loss
50,0.143100,0.133494
100,0.108300,0.123074
150,0.065900,0.128162
200,0.062300,0.123814


 TRAINING COMPLETE!


In [22]:
# Save final model
print(" Saving final model to Google Drive...")

model.save_pretrained(CONFIG['output_dir'])
tokenizer.save_pretrained(CONFIG['output_dir'])

print(f"\n Model saved to: {CONFIG['output_dir']}")

 Saving final model to Google Drive...

 Model saved to: /content/drive/MyDrive/wyckoff_chatbot/models/llama_wyckoff_lora


---
## STEP 6: Test the Model

In [23]:
# Test inference
def generate_response(question, max_new_tokens=256):
    """Generate response for a question."""
    prompt = f"""### Instruction:
You are an expert on Richard Wyckoff's trading methodology. Answer the following question accurately and thoroughly.

### Question:
{question}

### Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract answer part
    if "### Answer:" in response:
        response = response.split("### Answer:")[-1].strip()

    return response

In [24]:
# Test with sample questions
test_questions = [
    "What is a Spring in Wyckoff methodology?",
    "What are Wyckoff's three fundamental laws?",
    "When should I enter after seeing a Selling Climax?",
    "What is the difference between accumulation and distribution?",
    "How do I identify the Last Point of Support?"
]

print("="*60)
print(" TESTING FINE-TUNED MODEL")
print("="*60)

for q in test_questions:
    print(f"\n Q: {q}")
    print("-"*40)
    answer = generate_response(q)
    print(f" A: {answer[:500]}..." if len(answer) > 500 else f" A: {answer}")
    print()

 TESTING FINE-TUNED MODEL

 Q: What is a Spring in Wyckoff methodology?
----------------------------------------
 A: A Spring is a false breakdown below support that traps sellers, providing a low-risk entry opportunity for buyers.


 Q: What are Wyckoff's three fundamental laws?
----------------------------------------
 A: Wyckoff's three fundamental laws are: 1) Supply and Demand, 2) Effort and Result, and 3) Cause and Effect.


 Q: When should I enter after seeing a Selling Climax?
----------------------------------------
 A: Immediately if conditions allow, or wait for confirmation from subsequent events (like LPS).


 Q: What is the difference between accumulation and distribution?
----------------------------------------
 A: Accumulation represents smart money buying, while distribution represents smart money selling.


 Q: How do I identify the Last Point of Support?
----------------------------------------
 A: Identify LPS by tracking price action after a spring low, watching f

---
## STEP 7: Download Model (Optional)

In [ ]:
# Zip and download model (optional - it's already in Google Drive)
# Uncomment to download

# !zip -r /content/llama_wyckoff_lora.zip {CONFIG['output_dir']}
# from google.colab import files
# files.download('/content/llama_wyckoff_lora.zip')

---
## STEP 8: Build RAG Pipeline (Optional - Run after training)

In [29]:
!pip install -q sentence-transformers==2.2.2 chromadb==0.4.18


In [32]:
import sys
import gc

# Purge loaded modules to force reload from disk
pkgs = [p for p in sys.modules if p.startswith("transformers") or p.startswith("sentence_transformers")]
for p in pkgs:
    if p in sys.modules:
        del sys.modules[p]
gc.collect()

# Force install latest compatible versions to resolve file corruption
!pip install -q --upgrade transformers sentence-transformers accelerate --force-reinstall

# Re-import and run RAG pipeline
from sentence_transformers import SentenceTransformer
import numpy as np

# Fix for NumPy 2.0 compatibility (ChromaDB issue)
if not hasattr(np, 'float_'):
    np.float_ = np.float64

import chromadb

print(" Loading embedding model...")
embedder = SentenceTransformer('sentence-transformers/paraphrase-MiniLM-L6-v2')

# Create Chroma collection
client = chromadb.Client()

try:
    client.delete_collection("wyckoff_qa")
except:
    pass

collection = client.create_collection("wyckoff_qa")

# Add Q&A pairs
print(" Building vector index...")
questions = df['Questions'].tolist()
answers = df['Answers'].tolist()

embeddings = embedder.encode(questions, show_progress_bar=True)

collection.add(
    embeddings=embeddings.tolist(),
    documents=questions,
    metadatas=[{"answer": a} for a in answers],
    ids=[f"qa_{i}" for i in range(len(questions))]
)

print(f"\n RAG index built with {len(questions)} Q&A pairs!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
datasets 2.15.0 requires fsspec[http]<=2023.10.0,>=2023.1.0, but you have fsspec 2025.10.0 which is incompatible.
kubernetes 34.1.0 requires urllib3<2.4.0,>=1.24.2, but you have urllib3 2.5.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
google-adk 1.19.0 requires opentelemetry-api<=1.37.0,>=1.37.0, but you have opentelemetry-api 1.38.0 which is incompatible.
google-adk 1.19.0 requires opentelemetry-sdk<=1.37.0,>=1.37.0, but you have opentelemetry-sdk 1.38.0 which is incompatible.
torchaudio 2.9.0+cu126 requires torch==2.9.0, but you have torch 2.9.1 which is incompatible.
torchvision 0.24.0+cu126 requires torch==2.9.0, but you have torch 2.9.1 which is incompatible.
opentelemetry-

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


 Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


RuntimeError: Failed to import transformers.models.deta.configuration_deta because of the following error (look up to see its traceback):
No module named 'transformers.models.deta.configuration_deta'

In [ ]:
def search_similar_qas(query: str, top_k: int = 3):
    """
    Search for similar Q&As in the vector store.

    Args:
        query: User's question
        top_k: Number of similar Q&As to retrieve

    Returns:
        List of (question, answer, similarity_score) tuples
    """
    # Embed the query
    query_embedding = embedder.encode([query])[0]

    # Search in ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    # Format results
    output = []
    for i in range(len(results['documents'][0])):
        question = results['documents'][0][i]
        answer = results['metadatas'][0][i]['answer']
        distance = results['distances'][0][i]
        similarity = 1 - distance  # Convert distance to similarity
        output.append((question, answer, similarity))

    return output

# Test search
print("🧪 Testing RAG search...")
test_results = search_similar_qas("What is a Spring?", top_k=3)
print("\nTop 3 matches for 'What is a Spring?':")
for i, (q, a, sim) in enumerate(test_results, 1):
    print(f"\n{i}. [Similarity: {sim:.2f}]")
    print(f"   Q: {q[:80]}...")
    print(f"   A: {a[:100]}...")

In [ ]:
def ask_wyckoff(question: str, use_rag: bool = True, top_k: int = 3, max_tokens: int = 300):
    """
    Ask a question using RAG + Fine-tuned LLaMA.

    Args:
        question: User's question
        use_rag: Whether to use RAG (retrieval)
        top_k: Number of context items to retrieve
        max_tokens: Maximum tokens to generate

    Returns:
        answer: Generated answer
        context: Retrieved context (if use_rag=True)
    """
    context_str = ""
    retrieved_context = []

    # Step 1: Retrieve relevant context using RAG
    if use_rag:
        retrieved_context = search_similar_qas(question, top_k=top_k)

        for i, (q, a, sim) in enumerate(retrieved_context, 1):
            context_str += f"\nRelevant Information {i}:\nQ: {q}\nA: {a}\n"

    # Step 2: Build prompt with context
    if use_rag and context_str:
        prompt = f"""### Instruction:
You are an expert on Richard Wyckoff's trading methodology. Use the provided context to give a detailed, accurate answer.

{context_str}

### Question:
{question}

### Answer:
"""
    else:
        prompt = f"""### Instruction:
You are an expert on Richard Wyckoff's trading methodology. Provide a detailed, accurate answer.

### Question:
{question}

### Answer:
"""

    # Step 3: Generate response using fine-tuned model
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1
        )

    # Decode response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract answer part
    if "### Answer:" in response:
        response = response.split("### Answer:")[-1].strip()

    return response, retrieved_context

In [ ]:
test_questions = [
    "What is a Spring in Wyckoff methodology?",
    "When should I enter after seeing a Selling Climax?",
    "What are the five phases of accumulation?",
    "How do I identify distribution versus accumulation?",
    "What volume patterns indicate a successful Spring?"
]

for q in test_questions:
    print(f"\n{'='*60}")
    print(f" QUESTION: {q}")
    print("-" * 60)

    # Get answer with RAG
    answer, context = ask_wyckoff(q, use_rag=True, top_k=3)

    print(f"\n RETRIEVED CONTEXT:")
    for i, (cq, ca, sim) in enumerate(context, 1):
        print(f"   {i}. [{sim:.2f}] {cq[:50]}...")

    print(f"\n ANSWER:\n{answer}")

In [ ]:
print(" COMPARISON: With RAG vs Without RAG")
print("=" * 60)

test_q = "What is the difference between a Spring and a Shakeout?"

print(f"\n❓ Question: {test_q}\n")

# Without RAG
print("-" * 40)
print(" WITHOUT RAG (fine-tuned only):")
answer_no_rag, _ = ask_wyckoff(test_q, use_rag=False)
print(answer_no_rag)

# With RAG
print("-" * 40)
print(" WITH RAG (fine-tuned + retrieval):")
answer_with_rag, context = ask_wyckoff(test_q, use_rag=True)
print(answer_with_rag)

In [ ]:
import pickle
import os

# Create directory
rag_save_path = "/content/drive/MyDrive/wyckoff_chatbot/rag"
os.makedirs(rag_save_path, exist_ok=True)

# Save the Q&A data for rebuilding index later
df.to_csv(f"{rag_save_path}/wyckoff_qa_all.csv", index=False)

# Save embeddings (so we don't need to regenerate)
import numpy as np
np.save(f"{rag_save_path}/question_embeddings.npy", question_embeddings)

# Save metadata
metadata = {
    "embedding_model": "sentence-transformers/paraphrase-MiniLM-L6-v2",
    "num_documents": len(questions),
    "embedding_dim": question_embeddings.shape[1]
}
with open(f"{rag_save_path}/metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

print(f"✅ RAG components saved to: {rag_save_path}")
print(f"   - wyckoff_qa_all.csv")
print(f"   - question_embeddings.npy")
print(f"   - metadata.pkl")